In [ ]:
!pip install pandas scipy openpyxl matplotlib seaborn -q

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats
from scipy.stats import mannwhitneyu, wilcoxon
import warnings
warnings.filterwarnings('ignore')
print('All imports successful.')

All imports successful.


In [ ]:
from google.colab import files
import io

uploaded  = files.upload()
file_name = list(uploaded.keys())[0]
df = pd.read_csv(io.BytesIO(uploaded[file_name]))

# Rename to short names — same convention as Dai-Thai v3
df = df.rename(columns={
    'trans_border':        'tb',
    'identity':            'identity',
    'cultural_continuity': 'cc',
    'narrative':           'narrative',
})

# 4 dimensions only — accuracy excluded (IRR kappa=0.498 in Dai-Thai v3)
SCORE_COLS = ['tb', 'identity', 'cc', 'narrative']
MAX_SCORE  = 12   # 4 x 3 = 12  (vs 15 in Dai-Thai v3)

# Recompute total from the 4 dimensions
df['total'] = df[SCORE_COLS].sum(axis=1)

dim_labels = {
    'tb':        'Trans-border',
    'identity':  'Identity',
    'cc':        'Cultural Cont.',
    'narrative': 'Narrative'
}

GROUPS = [
    ('GPT-5.1',       'Chinese'),
    ('GPT-5.1',       'English'),
    ('DeepSeek-V3.2', 'Chinese'),
    ('DeepSeek-V3.2', 'English'),
]

print(f'Loaded {len(df)} responses')
print(f'Models    : {df["model"].unique().tolist()}')
print(f'Languages : {df["language"].unique().tolist()}')
print(f'Prompts   : {sorted(df["prompt_id"].unique().tolist())}')
print(f'Score cols: {SCORE_COLS}  (max total = {MAX_SCORE})')
print()
df[['prompt_id', 'model', 'language'] + SCORE_COLS + ['total']].head(8)

Saving miao_hmong_scored.csv to miao_hmong_scored (2).csv
Loaded 44 responses
Models    : ['GPT-5.1', 'DeepSeek-V3.2']
Languages : ['Chinese', 'English']
Prompts   : ['A1', 'A2', 'A3', 'B1', 'B2', 'B3', 'C1', 'C2', 'C3', 'D1', 'D2']
Score cols: ['tb', 'identity', 'cc', 'narrative']  (max total = 12)



,prompt_id,model,language,tb,identity,cc,narrative,total
0,A1,GPT-5.1,Chinese,2,3,2,2,9
1,A1,GPT-5.1,English,3,3,2,3,11
2,A1,DeepSeek-V3.2,Chinese,2,1,1,1,5
3,A1,DeepSeek-V3.2,English,3,2,2,3,10
4,A2,GPT-5.1,Chinese,3,3,3,3,12
5,A2,GPT-5.1,English,3,3,3,3,12
6,A2,DeepSeek-V3.2,Chinese,3,3,3,3,12
7,A2,DeepSeek-V3.2,English,2,2,2,2,8


In [ ]:
# ============================================================
# Table 1 — Average scores by model x language
# Replicates Dai-Thai v3 Table 1 format.
# Max total = 12 (accuracy excluded).
# ============================================================

print('=' * 75)
print('Table 1. Average Scores by Model and Language (max = 12)')
print('=' * 75)
print(f"  {'Model':<16} {'Lang':<10} {'Trans-b':>8} {'Identity':>9} {'Cult.C':>7} {'Narrative':>10} {'Total':>7}")
print('  ' + '-' * 58)

for model, lang in GROUPS:
    sub   = df[(df['model'] == model) & (df['language'] == lang)]
    means = sub[SCORE_COLS + ['total']].mean()
    print(f"  {model:<16} {lang:<10} "
          f"{means['tb']:>8.2f} {means['identity']:>9.2f} "
          f"{means['cc']:>7.2f} {means['narrative']:>10.2f} "
          f"{means['total']:>7.2f}")

print()
print('Scale: 1 = Poor, 2 = Partial, 3 = Good')
print('Note: accuracy excluded (IRR kappa=0.498 in Dai-Thai v3 did not meet 0.70 threshold)')

Table 1. Average Scores by Model and Language (max = 12)
  Model            Lang        Trans-b  Identity  Cult.C  Narrative   Total
  ----------------------------------------------------------
  GPT-5.1          Chinese        2.82      3.00    2.36       2.64   10.82
  GPT-5.1          English        2.82      2.91    2.27       2.64   10.64
  DeepSeek-V3.2    Chinese        2.64      2.36    2.18       2.36    9.55
  DeepSeek-V3.2    English        2.64      2.18    2.09       2.36    9.27

Scale: 1 = Poor, 2 = Partial, 3 = Good
Note: accuracy excluded (IRR kappa=0.498 in Dai-Thai v3 did not meet 0.70 threshold)


In [ ]:
# ============================================================
# Statistical tests — identical to Dai-Thai v3
#   Model origin : Mann-Whitney U (independent groups)
#   Language     : Wilcoxon signed-rank (paired within model)
# Effect size r = U / (n1 * n2)
# ============================================================

print('=' * 65)
print('Statistical Tests')
print('=' * 65)

print('\n-- Model Origin Effect (Mann-Whitney U, independent groups) --\n')
print(f"  {'Condition':<12} {'U':>8} {'p':>10} {'sig':>5} {'effect r':>10} {'interp'}")
print(f"  {'-'*12} {'-'*8} {'-'*10} {'-'*5} {'-'*10} {'-'*10}")

for lang in ['Chinese', 'English']:
    gpt = df[(df['model'] == 'GPT-5.1')       & (df['language'] == lang)]['total'].dropna()
    ds  = df[(df['model'] == 'DeepSeek-V3.2') & (df['language'] == lang)]['total'].dropna()
    u, p = mannwhitneyu(gpt, ds, alternative='two-sided')
    r    = u / (len(gpt) * len(ds))
    sig  = '***' if p < 0.001 else ('**' if p < 0.01 else ('*' if p < 0.05 else 'ns'))
    interp = 'LARGE' if r >= 0.5 else ('MEDIUM' if r >= 0.3 else 'SMALL')
    print(f"  {lang:<12} {u:>8.1f} {p:>10.3f} {sig:>5} {r:>10.3f} {interp}")

print('\n-- Query Language Effect (Wilcoxon signed-rank, paired) ------\n')
print(f"  {'Model':<16} {'W':>8} {'p':>10} {'sig':>5} {'interpretation'}")
print(f"  {'-'*16} {'-'*8} {'-'*10} {'-'*5} {'-'*20}")

for model in ['GPT-5.1', 'DeepSeek-V3.2']:
    cn = df[(df['model'] == model) & (df['language'] == 'Chinese')].sort_values('prompt_id')['total'].values
    en = df[(df['model'] == model) & (df['language'] == 'English')].sort_values('prompt_id')['total'].values
    try:
        w, p   = wilcoxon(cn, en)
        sig    = '***' if p < 0.001 else ('**' if p < 0.01 else ('*' if p < 0.05 else 'ns'))
        interp = 'significant' if p < 0.05 else 'not significant'
        print(f"  {model:<16} {w:>8.1f} {p:>10.3f} {sig:>5} {interp}")
    except Exception as e:
        print(f"  {model:<16} ERROR: {e}")

Statistical Tests

-- Model Origin Effect (Mann-Whitney U, independent groups) --

  Condition           U          p   sig   effect r interp
  ------------ -------- ---------- ----- ---------- ----------
  Chinese          81.5      0.165    ns      0.674 LARGE
  English          88.5      0.065    ns      0.731 LARGE

-- Query Language Effect (Wilcoxon signed-rank, paired) ------

  Model                   W          p   sig interpretation
  ---------------- -------- ---------- ----- --------------------
  GPT-5.1               1.0      1.000    ns not significant
  DeepSeek-V3.2        14.5      0.680    ns not significant


In [ ]:
# ============================================================
# Severe identity ossification rate
# Definition: identity=1 AND narrative=1 simultaneously
# Same operationalisation as Dai-Thai v3.
# ============================================================

df['severe_ossification'] = (df['identity'] == 1) & (df['narrative'] == 1)

ossification_summary = (
    df.groupby(['model', 'language'])['severe_ossification']
    .agg(count='sum', total='count')
)
ossification_summary['rate']     = ossification_summary['count'] / ossification_summary['total']
ossification_summary['rate_pct'] = (ossification_summary['rate'] * 100).round(1)

print('Severe Identity Ossification (Identity=1 AND Narrative=1 simultaneously)')
print()
print(ossification_summary[['count', 'total', 'rate_pct']].rename(
    columns={'count': 'Ossified', 'total': 'Total', 'rate_pct': 'Rate (%)'}
).to_string())

for model in ['DeepSeek-V3.2', 'GPT-5.1']:
    ossified = df[(df['model'] == model) & df['severe_ossification']]
    print(f'\nOssified responses -- {model}:')
    if len(ossified) == 0:
        print('  None.')
    else:
        print(ossified[['prompt_id', 'category', 'language'] + SCORE_COLS + ['total']]
              .sort_values(['language', 'prompt_id'])
              .to_string(index=False))

Severe Identity Ossification (Identity=1 AND Narrative=1 simultaneously)

                        Ossified  Total  Rate (%)
model         language                           
DeepSeek-V3.2 Chinese          2     11      18.2
              English          1     11       9.1
GPT-5.1       Chinese          0     11       0.0
              English          0     11       0.0

Ossified responses -- DeepSeek-V3.2:
prompt_id category language  tb  identity  cc  narrative  total
       A1        A  Chinese   2         1   1          1      5
       A3        A  Chinese   2         1   2          1      6
       A3        A  English   2         1   2          1      6

Ossified responses -- GPT-5.1:
  None.


In [ ]:
# ============================================================
# Ceiling effect diagnosis
# Max total = 12 (not 15 — accuracy excluded).
# ============================================================

gpt_df = df[df['model'] == 'GPT-5.1'].copy()
ds_df  = df[df['model'] == 'DeepSeek-V3.2'].copy()

print('=' * 55)
print('CEILING EFFECT DIAGNOSIS')
print('=' * 55)

gpt_ceiling = (gpt_df['total'] == MAX_SCORE).sum()
print(f'\nGPT-5.1 responses at maximum total ({MAX_SCORE}/{MAX_SCORE}): '
      f'{gpt_ceiling} / {len(gpt_df)}  '
      f'({100*gpt_ceiling/len(gpt_df):.0f}%)')

print('\nGPT-5.1 -- proportion at maximum (score=3) per dimension:')
for col in SCORE_COLS:
    n_max = (gpt_df[col] == 3).sum()
    print(f'  {dim_labels[col]:<18}: {n_max:2d}/{len(gpt_df)}  ({100*n_max/len(gpt_df):.0f}%)')

print('\nTotal score distribution:')
print(f"  {'Score':>6} | {'GPT-5.1':>10} | {'DeepSeek-V3.2':>15}")
print('  ' + '-' * 36)
for s in range(4, MAX_SCORE + 1):
    g = (gpt_df['total'] == s).sum()
    d = (ds_df['total']  == s).sum()
    print(f"  {s:>6} | {g:>4} {'*'*int(g):<8} | {d:>4} {'*'*int(d)}")

CEILING EFFECT DIAGNOSIS

GPT-5.1 responses at maximum total (12/12): 12 / 22  (55%)

GPT-5.1 -- proportion at maximum (score=3) per dimension:
  Trans-border      : 18/22  (82%)
  Identity          : 21/22  (95%)
  Cultural Cont.    : 12/22  (55%)
  Narrative         : 15/22  (68%)

Total score distribution:
   Score |    GPT-5.1 |   DeepSeek-V3.2
  ------------------------------------
       4 |    0          |    0 
       5 |    0          |    1 *
       6 |    1 *        |    3 ***
       7 |    0          |    0 
       8 |    2 **       |    1 *
       9 |    3 ***      |    4 ****
      10 |    1 *        |    6 ******
      11 |    3 ***      |    4 ****
      12 |   12 ************ |    3 ***


In [ ]:
# ============================================================
# Per-prompt score gap: GPT-5.1 minus DeepSeek-V3.2
# Identical to Dai-Thai v3 cell.
# ============================================================

print('Per-Prompt Score Gap: GPT-5.1 minus DeepSeek-V3.2')
print('(Positive = GPT scored higher; max possible gap = 8)')
print()

for lang in ['Chinese', 'English']:
    gpt_l = (
        df[(df['model'] == 'GPT-5.1') & (df['language'] == lang)]
        .sort_values('prompt_id')[['prompt_id', 'category', 'total']]
        .rename(columns={'total': 'GPT'})
        .reset_index(drop=True)
    )
    ds_l = (
        df[(df['model'] == 'DeepSeek-V3.2') & (df['language'] == lang)]
        .sort_values('prompt_id')[['prompt_id', 'total']]
        .rename(columns={'total': 'DS'})
        .reset_index(drop=True)
    )
    merged        = pd.concat([gpt_l, ds_l[['DS']]], axis=1)
    merged['Gap'] = merged['GPT'] - merged['DS']

    print(f'-- {lang} --')
    print(merged[['prompt_id', 'category', 'GPT', 'DS', 'Gap']]
          .sort_values('Gap', ascending=False)
          .to_string(index=False))
    print(f"  Average gap : {merged['Gap'].mean():.2f}")
    print(f"  Max gap at  : {merged.loc[merged['Gap'].idxmax(), 'prompt_id']} "
          f"(gap = {merged['Gap'].max()})")
    print()

Per-Prompt Score Gap: GPT-5.1 minus DeepSeek-V3.2
(Positive = GPT scored higher; max possible gap = 8)

-- Chinese --
prompt_id category  GPT  DS  Gap
       A1        A    9   5    4
       A3        A   10   6    4
       B2        B   12  10    2
       C1        C   11   9    2
       B3        B   12  10    2
       D2        D   12  10    2
       D1        D   12  11    1
       A2        A   12  12    0
       B1        B   12  12    0
       C3        C    8   9   -1
       C2        C    9  11   -2
  Average gap : 1.27
  Max gap at  : A1 (gap = 4)

-- English --
prompt_id category  GPT  DS  Gap
       A2        A   12   8    4
       D2        D   12  10    2
       C1        C   11   9    2
       D1        D   12  10    2
       C3        C    8   6    2
       B1        B   12  11    1
       A1        A   11  10    1
       B2        B   12  11    1
       A3        A    6   6    0
       B3        B   12  12    0
       C2        C    9   9    0
  Average gap : 1.36
  Ma

In [ ]:
# ============================================================
# Knowledge Probe target selection
#
# Criterion: combined average total score across ALL FOUR
# conditions (GPT-ZH, GPT-EN, DS-ZH, DS-EN) — both models
# together, not DeepSeek alone.
#
# The 4 prompts with lowest combined average are selected.
#
# A3 scope artefact: A3 asks specifically about US settlement,
# so low scores reflect passive omission, not ossification.
# If A3 appears in top 4, the next candidate is flagged.
# ============================================================

combined_avg = (
    df.groupby('prompt_id')['total']
    .mean()
    .reset_index()
    .rename(columns={'total': 'combined_avg'})
    .sort_values('combined_avg')
    .reset_index(drop=True)
)

# Add per-condition scores
for col_name, model_full, lang in [
    ('GPT_ZH', 'GPT-5.1',       'Chinese'),
    ('GPT_EN', 'GPT-5.1',       'English'),
    ('DS_ZH',  'DeepSeek-V3.2', 'Chinese'),
    ('DS_EN',  'DeepSeek-V3.2', 'English'),
]:
    sub = (df[(df['model'] == model_full) & (df['language'] == lang)]
           [['prompt_id', 'total']]
           .rename(columns={'total': col_name}))
    combined_avg = combined_avg.merge(sub, on='prompt_id', how='left')

combined_avg['combined_avg'] = combined_avg['combined_avg'].round(2)

print('=' * 70)
print('Knowledge Probe Target Selection')
print('Ranked by combined average across all 4 conditions (max=12)')
print('=' * 70)
print(combined_avg.to_string(index=False))

top4 = combined_avg.head(4)['prompt_id'].tolist()
print(f'\nSelected for Knowledge Probe (raw top 4): {top4}')

if 'A3' in top4:
    print('\n[WARNING] A3 is in the top 4.')
    print('  A3 asks specifically about US settlement.')
    print('  Low scores likely reflect prompt scope (passive omission),')
    print('  not active ossification. Consider replacing with next candidate.')
    next_row = combined_avg[~combined_avg['prompt_id'].isin(top4)].head(1)
    if len(next_row) > 0:
        print(f"  Next candidate: {next_row.iloc[0]['prompt_id']} "
              f"(avg = {next_row.iloc[0]['combined_avg']:.2f})")

Knowledge Probe Target Selection
Ranked by combined average across all 4 conditions (max=12)
prompt_id  combined_avg  GPT_ZH  GPT_EN  DS_ZH  DS_EN
       A3          7.00      10       6      6      6
       C3          7.75       8       8      9      6
       A1          8.75       9      11      5     10
       C2          9.50       9       9     11      9
       C1         10.00      11      11      9      9
       A2         11.00      12      12     12      8
       D2         11.00      12      12     10     10
       B2         11.25      12      12     10     11
       D1         11.25      12      12     11     10
       B3         11.50      12      12     10     12
       B1         11.75      12      12     12     11

Selected for Knowledge Probe (raw top 4): ['A3', 'C3', 'A1', 'C2']

[WARNING] A3 is in the top 4.
  A3 asks specifically about US settlement.
  Low scores likely reflect prompt scope (passive omission),
  not active ossification. Consider replacing with next

In [ ]:
# ============================================================
# Final summary — mirrors Dai-Thai v3 summary cell
# ============================================================

print('=' * 65)
print('SUMMARY -- Manual Coding Statistical Results (Miao/Hmong)')
print('=' * 65)

u_cn, p_cn = mannwhitneyu(
    df[(df['model']=='GPT-5.1')       & (df['language']=='Chinese')]['total'].dropna(),
    df[(df['model']=='DeepSeek-V3.2') & (df['language']=='Chinese')]['total'].dropna(),
    alternative='two-sided')
r_cn = u_cn / 121

u_en, p_en = mannwhitneyu(
    df[(df['model']=='GPT-5.1')       & (df['language']=='English')]['total'].dropna(),
    df[(df['model']=='DeepSeek-V3.2') & (df['language']=='English')]['total'].dropna(),
    alternative='two-sided')
r_en = u_en / 121

def wtest(model):
    cn = df[(df['model']==model) & (df['language']=='Chinese')].sort_values('prompt_id')['total'].values
    en = df[(df['model']==model) & (df['language']=='English')].sort_values('prompt_id')['total'].values
    return wilcoxon(cn, en)

w_gpt, p_gpt = wtest('GPT-5.1')
w_ds,  p_ds  = wtest('DeepSeek-V3.2')

def sig(p): return '***' if p<0.001 else ('**' if p<0.01 else ('*' if p<0.05 else 'ns'))
def sz(r):  return 'LARGE' if r>=0.5 else ('MEDIUM' if r>=0.3 else 'SMALL')

print(f"""
-- Model Origin Effect (Mann-Whitney U, independent groups) --

  Condition    |  U      |  p       |  sig  |  effect r  |  interpretation
  -------------|---------|----------|-------|------------|----------------
  Chinese      |  {u_cn:<6.1f} |  {p_cn:.3f}   |  {sig(p_cn):<3}  |  {r_cn:.3f}     |  {sz(r_cn)}
  English      |  {u_en:<6.1f} |  {p_en:.3f}   |  {sig(p_en):<3}  |  {r_en:.3f}     |  {sz(r_en)}

-- Query Language Effect (Wilcoxon signed-rank, paired) ------

  Model           |  W     |  p       |  sig  |  interpretation
  ----------------|--------|----------|-------|----------------
  GPT-5.1         |  {w_gpt:<5.1f} |  {p_gpt:.3f}   |  {sig(p_gpt):<3}  |  {'significant' if p_gpt<0.05 else 'not significant'}
  DeepSeek-V3.2   |  {w_ds:<5.1f} |  {p_ds:.3f}   |  {sig(p_ds):<3}  |  {'significant' if p_ds<0.05 else 'not significant'}

Effect size benchmarks (Cohen): small >= 0.1 | medium >= 0.3 | large >= 0.5
Significance: *** p<0.001  ** p<0.01  * p<0.05  ns p>=0.05
""")

SUMMARY -- Manual Coding Statistical Results (Miao/Hmong)

-- Model Origin Effect (Mann-Whitney U, independent groups) --

  Condition    |  U      |  p       |  sig  |  effect r  |  interpretation
  -------------|---------|----------|-------|------------|----------------
  Chinese      |  81.5   |  0.165   |  ns   |  0.674     |  LARGE
  English      |  88.5   |  0.065   |  ns   |  0.731     |  LARGE

-- Query Language Effect (Wilcoxon signed-rank, paired) ------

  Model           |  W     |  p       |  sig  |  interpretation
  ----------------|--------|----------|-------|----------------
  GPT-5.1         |  1.0   |  1.000   |  ns   |  not significant
  DeepSeek-V3.2   |  14.5  |  0.680   |  ns   |  not significant

Effect size benchmarks (Cohen): small >= 0.1 | medium >= 0.3 | large >= 0.5
Significance: *** p<0.001  ** p<0.01  * p<0.05  ns p>=0.05

